In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
# Load the dataset
data = pd.read_csv('king_county_house_pandas_cleaned.csv')
data

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price,house_age
0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,3,7,...,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,2014-10-13,221900.0,71
1,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3,7,...,1951,1991.0,98125,47.7210,-122.319,1690.0,7639.0,2014-12-09,538000.0,75
2,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3,6,...,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,2015-02-25,180000.0,93
3,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5,7,...,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,2014-12-09,604000.0,61
4,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3,8,...,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,2015-02-18,510000.0,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,263000018,3.0,2.50,1530.0,1131.0,3.0,0.0,0.0,3,8,...,2009,0.0,98103,47.6993,-122.346,1530.0,1509.0,2014-05-21,360000.0,17
21593,6600060120,4.0,2.50,2310.0,5813.0,2.0,0.0,0.0,3,8,...,2014,0.0,98146,47.5107,-122.362,1830.0,7200.0,2015-02-23,400000.0,12
21594,1523300141,2.0,0.75,1020.0,1350.0,2.0,0.0,0.0,3,7,...,2009,0.0,98144,47.5944,-122.299,1020.0,2007.0,2014-06-23,402101.0,17
21595,291310100,3.0,2.50,1600.0,2388.0,2.0,NaN,0.0,3,8,...,2004,0.0,98027,47.5345,-122.069,1410.0,1287.0,2015-01-16,400000.0,22


In [31]:
data_wf_geo= data.copy()

# 1.Aufteilen in Häuser mit bekanntem Waterfront vs. Häuser mit NaN
known = data_wf_geo[data_wf_geo['waterfront'].notna()]
missing = data_wf_geo[data_wf_geo['waterfront'].isna()]

if not missing.empty:
    # 2. Den Classifier trainieren (wir schauen nur auf die 1 nächsten Nachbarn)
    # n_neighbors=1 sorgt dafür, dass wir exakt den Wert des nächsten Hauses kopieren
    knn = KNeighborsClassifier(n_neighbors=1)
    
    # Wir trainieren auf Breitengrad und Längengrad
    knn.fit(known[['lat', 'long']], known['waterfront'])
    
    # 3. Die fehlenden Werte vorhersagen
    predictions = knn.predict(missing[['lat', 'long']])
    
    # 4. Die Werte im Original-DataFrame füllen
    data_wf_geo.loc[data_wf_geo['waterfront'].isna(), 'waterfront'] = predictions

print(f"Fertig! Verbleibende NaNs: {data_wf_geo['waterfront'].isna().sum()}")

Fertig! Verbleibende NaNs: 0


In [32]:
data_wf_zip= data.copy()

# Wir gruppieren nach zip_code und füllen die NaNs mit dem Modus (häufigster Wert)
data_wf_zip['waterfront'] = data_wf_zip.groupby('zipcode')['waterfront'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
)

In [33]:
# Nur die Zeilen anzeigen, die sich UNTERSCHEIDEN
unterschiede = data_wf_geo[data_wf_geo['waterfront'] != data_wf_zip['waterfront']]
print(f"Anzahl der Abweichungen: {len(unterschiede)}")
unterschiede[['lat', 'long', 'zipcode', 'waterfront']].head(20)

Anzahl der Abweichungen: 12


,lat,long,zipcode,waterfront
1482,47.4781,-122.490,98070,1.0
1612,47.5387,-122.265,98118,1.0
5711,47.5789,-122.076,98075,1.0
5929,47.4003,-122.422,98070,1.0
8969,47.5684,-122.060,98075,1.0
10349,47.5007,-122.223,98178,1.0
13027,47.4096,-122.449,98070,1.0
13299,47.3905,-122.462,98070,1.0
15017,47.5188,-122.256,98118,1.0
17966,47.5648,-122.210,98040,1.0


In [34]:
##NAN-Werte in sqft_basement füllen
data['sqft_basement']=data['sqft_basement'].fillna(data['sqft_living']-data['sqft_above'])
#Gegengeprüft: Gibt es jetzt Häuser mit negativem Basement?
negative_basement = data[data['sqft_basement'] < 0]
print(f"Anzahl der Häuser mit negativem Basement: {len(negative_basement)}")

Anzahl der Häuser mit negativem Basement: 0


In [35]:
#NaNs in 'yr_renovated' mit 0 füllen, da 0 bedeutet, dass das Haus nie renoviert wurde
data['yr_renovated'] = data['yr_renovated'].fillna(0)

#neue Spalte 'is_renovated' erstellen, die 1 ist, wenn das Haus renoviert wurde (yr_renovated > 0) und 0 ist nicht renoviert
data['is_renovated'] = (data['yr_renovated'] > 0).astype(int)

#waterfront NAN mit sk-learn von oben füllen
data['waterfront']=data_wf_geo['waterfront']

#view NAN füllen mit Nachbarn im gleichen Zip-Code
data['view']=data.groupby('zipcode')['view'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0))

data.isna().sum()   


id               0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
date             0
price            0
house_age        0
is_renovated     0
dtype: int64

In [36]:
data.to_csv('king_county_house_pandas_filled.csv', index=False)